# EP2 - IT Assist Multi-Agent

## Pipeline jerárquico con Manager

Este notebook demuestra un flujo donde un Manager de CrewAI coordina agentes especializados:

> **Usuario → Manager → Investigador / Analista / Comunicador → Usuario**

### Objetivo
Comparar la orquestaci?n jerárquica con el flujo secuencial y evidenciar delegación adaptativa.

### Indicadores demostrados
- **IE1-IE2:** herramientas y CrewAI.
- **IE4:** recuperación RAG.
- **IE5:** planificación dinámica.
- **IE7-IE9:** decisiones, arquitectura y justificación.

In [ ]:
"""
EP2 - IT Assist Multi-Agent Orchestration
Pipeline JERÁRQUICO: Manager delega -> Investigador / Analista / Comunicador

Arquitectura:
    Manager (LLM):    Decide quien hace que, evalua calidad, re-delega si hace falta.
    Agente 1:         Investigador (con tool RAG)
    Agente 2:         Analista (sintetiza)
    Agente 3:         Comunicador (redacta respuesta final)

Proceso: Hierarchical (el manager orquesta dinamicamente)
LLM:     Ollama local (llama3.2:3b / qwen3.8:27b) para todos
Embeddings RAG: Ollama (nomic-embed-text)

Diferencia vs Sequential:
    - Sequential: orden fijo A->B->C, sin evaluacion intermedia.
    - Hierarchical: el Manager puede pedir re-hacer, cambiar orden,
      o agregar pasos según la calidad de cada output.

Uso:
    Ejecuta las celdas de este notebook para probar el pipeline jerárquico.
"""

import os
import sys
import time
import re

from dotenv import load_dotenv
from loguru import logger

load_dotenv()

logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss.SSS}</green> | <level>{level: <8}</level> | <level>{message}</level>",
    level="INFO",
)

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from rag_tool import buscar_base_conocimiento


## 1. Configuración y arquitectura

El pipeline utiliza el mismo LLM, vectorstore y herramientas de IT Assist. La diferencia es `Process.hierarchical` con `manager_llm`.

```text
Consulta
   →
Manager: interpreta y delega
   → Investigador: recupera fuentes
   → Analista: prioriza y sintetiza
   → Comunicador: redacta
   →
Respuesta final
```

In [ ]:
def _llm_kwargs() -> dict:
    return dict(
        model=os.getenv("LLM_MODEL", "llama3.2:3b"),
        api_key=os.getenv("LLM_API_KEY", "ollama"),
        base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
        temperature=0.3,
    )


def _manager_llm_kwargs() -> dict:
    """Manager con temperatura ligeramente más alta para decidir mejor."""
    kw = _llm_kwargs()
    kw["temperature"] = 0.4
    return kw


@tool("buscar_base_conocimiento_it")
def buscar_base_conocimiento_it(consulta: str) -> str:
    """
    Busca en la base de conocimiento de IT y retorna fragmentos relevantes.

    Argumentos:
        consulta: La pregunta o tema a buscar.
    """
    return buscar_base_conocimiento(consulta)


@tool("clasificar_prioridad_ticket")
def clasificar_prioridad_ticket(descripcion: str) -> str:
    """
    Clasifica la prioridad de un incidente IT según la descripción.

    Argumentos:
        descripcion: Descripción del problema o contexto.
    """
    texto = descripcion.lower()
    criticos = ["no puedo trabajar", "caido", "bloqueado", "emergencia", "critico"]
    altos = ["vpn", "correo", "no funciona", "falla", "error"]
    if any(kw in texto for kw in criticos):
        return "PRIORIDAD: CRITICA (P1) - Impacto total. SLA: 1 hora."
    if any(kw in texto for kw in altos):
        return "PRIORIDAD: ALTA (P2) - Impacto parcial. SLA: 4 horas."
    return "PRIORIDAD: MEDIA (P3) - Impacto menor. SLA: 24 horas."


## 2. Agentes, herramientas y tareas

| Rol | Funci?n | Evidencia |
|---|---|---|
| Manager | Planificar y delegar | `manager_llm` |
| Investigador | Recuperar fuentes | Tool RAG |
| Analista | Clasificar prioridad | Tool de prioridad |
| Comunicador | Redactar respuesta | Salida final |

Las tareas declaran contexto compartido, mientras el Manager coordina la ejecución.

In [ ]:
def _build_agents():
    investigador = Agent(
        role="Investigador de Base de Conocimiento IT",
        goal="Encontrar en la base vectorial los fragmentos más relevantes con su fuente.",
        backstory=(
            "Eres especialista en búsqueda semantica. Usas un índice FAISS "
            "con 7 documentos de IT. Tu trabajo es encontrar la información "
            "exacta y no inventar nada."
        ),
        tools=[buscar_base_conocimiento_it],
        llm=_llm_kwargs(),
        verbose=True,
        allow_delegation=False,
        max_iter=2,
    )
    analista = Agent(
        role="Analista Tecnico de Resoluciones IT",
        goal="Sintetizar hallazgos: procedimiento, prioridad, pasos exactos, fuentes.",
        backstory=(
            "Eres analista de soporte TI. Extraes el procedimiento aplicable, "
            "clasificas prioridad y estructuras la resolución técnica."
        ),
        tools=[clasificar_prioridad_ticket],
        llm=_llm_kwargs(),
        verbose=True,
        allow_delegation=False,
        max_iter=2,
    )
    comunicador = Agent(
        role="Redactor de Comunicacion al Usuario Final",
        goal="Redactar la respuesta final en tono profesional con pasos, fuentes y escalamiento.",
        backstory=(
            "Traduces análisis técnico a lenguaje accesible. "
            "Respuesta breve, accionable, con fuentes y opción de ticket."
        ),
        tools=[],
        llm=_llm_kwargs(),
        verbose=True,
        allow_delegation=False,
        max_iter=1,
    )
    return investigador, analista, comunicador


def _build_tasks(consulta, ag_inv, ag_an, ag_co):
    t_inv = Task(
        description=(
            f"Consulta: {consulta}\n\n"
            "Usa la tool de búsqueda para encontrar fragmentos relevantes. "
            "Retorna texto con fuentes [archivo | página | tipo]."
        ),
        expected_output="Fragmentos relevantes con fuentes citadas.",
        agent=ag_inv,
    )
    t_an = Task(
        description=(
            "Recibe hallazgos. Identifica procedimiento, clasifica prioridad "
            "(usa la tool de clasificacion), extrae pasos y cita fuentes. "
            "Retorna análisis estructurado."
        ),
        expected_output="Análisis con procedimiento, prioridad, pasos y fuentes.",
        agent=ag_an,
        context=[t_inv],
    )
    t_co = Task(
        description=(
            "Recibe análisis. Redacta la respuesta final al usuario: "
            "saludo, pasos numerados, fuente, canal de escalamiento, cierre. "
            "Tono profesional y empático. No inventes información."
        ),
        expected_output="Respuesta final lista para enviar al usuario.",
        agent=ag_co,
        context=[t_inv, t_an],
    )
    return t_inv, t_an, t_co


## 3. Validación y métricas

La respuesta se valida por pasos, fuentes y escalamiento. También se registran tiempo total y tiempo acumulado por tarea.

El fallback permite identificar salidas incompletas en lugar de ocultar el problema. Esta evidencia es útil para evaluar autonomía y calidad del agente.

In [ ]:
def _validate_response(response: str) -> dict:
    """Valida campos clave de la respuesta final."""
    text_lower = response.lower()
    checks = {
        "pasos": bool(re.search(r"\d+[\.)]\s", response)) or "paso" in text_lower,
        "fuente": any(k in text_lower for k in ["fuente", "archivo", "pagina", "pág"]),
        "escalamiento": any(k in text_lower for k in ["ticket", "mesa de ayuda", "escal"]),
    }
    missing = [key for key, present in checks.items() if not present]
    return {"valid": not missing, "checks": checks, "missing": missing}


def _get_fallback(consulta: str, missing: list[str]) -> str:
    """Genera una respuesta de respaldo cuando faltan campos."""
    return (
        f"⚠️ **Atención**: La respuesta no incluyó: {', '.join(missing)}.\n\n"
        f"Para tu consulta *\"{consulta}\"*:\n\n"
        "1. Revisa la base de conocimiento interna.\n"
        "2. Si el problema persiste, abre un ticket en **Mesa de Ayuda**.\n"
        "3. Contacta a soporte@empresa.cl si es crítico.\n"
    )


## 4. Ejecución y decisiones adaptativas

Se recomienda probar dos escenarios:

1. **Consulta simple:** contraseña olvidada.
2. **Consulta ambigua o crítica:** VPN con error de autenticación.

| Criterio | Secuencial | Jerárquico |
|---|---|---|
| Orden | Fijo | Coordinado por Manager |
| Planificaci?n | Predefinida | Dinámica |
| Reevaluaci?n | Limitada | Posible mediante delegación |
| Costo | Menor | Mayor |

In [ ]:
def run_pipeline(consulta: str) -> dict:
    """Ejecuta el pipeline jerárquico con validación y métricas."""
    logger.info(f"[PIPELINE] Pipeline jerárquico iniciado | Consulta: {consulta[:60]}")
    t_start = time.time()
    ag_inv, ag_an, ag_co = _build_agents()
    t_inv, t_an, t_co = _build_tasks(consulta, ag_inv, ag_an, ag_co)
    agent_timings: dict[str, float] = {}
    previous_task_end = t_start

    def on_task_end(output):
        nonlocal previous_task_end
        name = getattr(output, "agent", None) or "desconocido"
        now = time.time()
        agent_timings[name] = round(now - previous_task_end, 2)
        previous_task_end = now
        logger.info(f"[TASK] {name} completado | Output: {len(str(output))} chars")

    crew = Crew(
        agents=[ag_inv, ag_an, ag_co],
        tasks=[t_inv, t_an, t_co],
        process=Process.hierarchical,
        manager_llm=_manager_llm_kwargs(),
        verbose=False,
        task_callback=on_task_end,
    )
    result = crew.kickoff(inputs={"consulta": consulta})
    raw_result = result.raw if result else "N/A"
    validation = _validate_response(raw_result)
    total_time = round(time.time() - t_start, 2)
    if not validation["valid"]:
        logger.warning(f"[VALIDATION] Faltan campos: {validation['missing']}")
    logger.info(f"[PIPELINE] Pipeline jerárquico completado en {total_time}s")
    return {
        "consulta": consulta,
        "resultado": raw_result,
        "validacion": validation,
        "latencia_total_s": total_time,
        "latencia_por_agente": agent_timings,
    }


## 5. Resultado y conclusiones

La estrategia jerárquica es adecuada para consultas con varias etapas, prioridades o condiciones cambiantes. Agrega costo y complejidad, pero ofrece mayor flexibilidad de coordinaci?n.

La elección técnica debe depender del requerimiento: secuencial para flujos estables y jerárquico para decisiones adaptativas.

In [ ]:
def main():
    if len(sys.argv) > 1:
        consulta = " ".join(sys.argv[1:])
    else:
        consulta = "La VPN me dice autenticación fallida, que reviso primero?"

    print("=" * 70)
    print("EP2 - IT Assist Multi-Agent | Pipeline JERÁRQUICO")
    print("=" * 70)
    print(f"\nConsulta: {consulta}\n")

    result = run_pipeline(consulta)

    print("\n" + "=" * 70)
    print("RESULTADO FINAL")
    print("=" * 70)
    print(result["resultado"])


if __name__ == "__main__":
    main()
